In [41]:
import os
import cv2
import glob
import numpy as np
import pandas as pd
from astropy.io import fits 
from skimage.transform import resize   
from sunpy.map.header_helper import make_heliographic_header
from skimage.draw import rectangle
import Sdo_image_processing as sdopro


In [42]:
project_log_scale_stat_df = pd.read_csv('../DATASET/stonyhurst_aia_statistics.csv')
project_log_scale_stat_df.head()

,date,mean_193,std_193,mean_171,std_171,mean_304,std_304
0,20100513,5.891195,0.662655,6.227106,0.486360,4.799089,0.399771
1,20100514,5.888975,0.658746,6.217736,0.480624,4.795589,0.390057
2,20100515,5.902561,0.681372,6.227304,0.460371,4.798674,0.387924
3,20100516,5.861914,0.711962,6.192952,0.476851,4.790066,0.396454
4,20100517,5.854000,0.742108,6.161634,0.502720,4.790983,0.405375


In [43]:
Average_log_scaled_mean_193 = np.mean(project_log_scale_stat_df['mean_193'])
Average_log_scaled_std_193 = np.mean(project_log_scale_stat_df['std_193'])
Average_log_scaled_mean_171 = np.mean(project_log_scale_stat_df['mean_171'])
Average_log_scaled_std_171 = np.mean(project_log_scale_stat_df['std_171'])
Average_log_scaled_mean_304 = np.mean(project_log_scale_stat_df['mean_304'])
Average_log_scaled_std_304 = np.mean(project_log_scale_stat_df['std_304'])

In [ ]:

years = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
for year in years:
    path_193 = f"../DATASET/AIA/193/{year}/*.*"
    path_171 = f"../DATASET/AIA/171/{year}/*.*"
    path_304 = f"../DATASET/AIA/304/{year}/*.*"

    files_193 = sorted(glob.glob(path_193))
    files_171 = sorted(glob.glob(path_171))
    files_304 = sorted(glob.glob(path_304))
   
    # Build dicts
    aia_dict_193 = {os.path.basename(f)[-26:-15]: f for f in files_193}
    aia_dict_171 = {os.path.basename(f)[-26:-15]: f for f in files_171}
    aia_dict_304 = {os.path.basename(f)[-26:-15]: f for f in files_304}
    
    common_timestamps = sorted(
        set(aia_dict_193.keys()) &
        set(aia_dict_171.keys()) &
        set(aia_dict_304.keys())
    )

    print(f"\nYear {year}: Found {len(common_timestamps)} matched dates")

    for i, ts in enumerate(common_timestamps, 1):
        aia_file_193 = aia_dict_193[ts]
        aia_file_171 = aia_dict_171[ts]
        aia_file_304 = aia_dict_304[ts]

        
        print("\n ***********STANDARDIZATION PROCESSING****************\n")
        
        print(f"({i:03d}) {ts} — "
              f"AIA 193: {os.path.basename(aia_file_193)} | "
              f"AIA 171: {os.path.basename(aia_file_171)} | "
              f"AIA 304: {os.path.basename(aia_file_304)}")

        ### PROJECTED STONYHURST
        
        standardized_projected_193, binary_p = sdopro.Image_processing(aia_file_193, Average_log_scaled_mean_193, Average_log_scaled_std_193, binary_operate=True)
        standardized_projected_171, _ = sdopro.Image_processing(aia_file_171, Average_log_scaled_mean_171, Average_log_scaled_std_171)
        standardized_projected_304, _ = sdopro.Image_processing(aia_file_304, Average_log_scaled_mean_304, Average_log_scaled_std_304)


        three_channel_projected = np.stack((standardized_projected_193, 
                                        standardized_projected_171, 
                                        standardized_projected_304), axis=-1)

        three_channel_projected_float = three_channel_projected.astype(np.float32)
        # print(ts)

        output_path = f"../DATASET/stonyhurst_three_channel/{ts}"
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        with fits.HDUList([fits.PrimaryHDU(three_channel_projected_float)]) as hdul:
            hdul.writeto(output_path, overwrite=True)
        
        binary_int = binary_p.astype(int)
        hdu_binary = fits.PrimaryHDU(binary_int)
        hdul_binary_fit = fits.HDUList([hdu_binary])

        # # # Write the FITS file to disk
        hdul_binary_fit.writeto(f"../DATASET/binary_map/{ts}", overwrite=True)
        

            
            

        